# Finetuning実験 Part 2: QLoRAファインチューニング

**目的**: Qwen2.5-7B-InstructをPOI質問応答タスクでファインチューニング

**入力**:
- `data/finetuning_train.json` - 学習データ
- `data/finetuning_valid.json` - 検証データ

**出力**:
- `models/qwen2.5-7b-shibuya-poi/` - ファインチューニング済みLoRAアダプタ

**実行環境**: Google Colab T4 GPU（16GB VRAM）

**作成日**: 2026-01-28

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q torch transformers accelerate bitsandbytes
!pip install -q peft trl datasets
!pip install -q pandas numpy tqdm matplotlib
print("パッケージインストール完了")

In [ ]:
# 1.2 GPU確認・メモリ管理
import torch
import gc

def print_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"GPU VRAM: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    import psutil
    print(f"RAM: {psutil.virtual_memory().percent}%")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
print_memory()

In [ ]:
# 1.3 Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
MODEL_DIR = f"{BASE_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)
sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.4 設定
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = f"{MODEL_DIR}/qwen2.5-7b-shibuya-poi"

# QLoRA設定
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# 学習設定
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024

print(f"Model: {MODEL_NAME}")
print(f"Output: {OUTPUT_DIR}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Training: epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRADIENT_ACCUMULATION}")

## Section 2: データ読み込み

In [ ]:
# 2.1 学習データ読み込み
import json
from datasets import Dataset

with open(f"{DATA_DIR}/finetuning_train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(f"{DATA_DIR}/finetuning_valid.json", "r", encoding="utf-8") as f:
    valid_data = json.load(f)

print(f"学習データ: {len(train_data)}件")
print(f"検証データ: {len(valid_data)}件")

# サンプル確認
print("\nサンプル:")
sample = train_data[0]
print(f"  instruction: {sample['instruction'][:50]}...")
print(f"  output: {sample['output'][:50]}...")

In [ ]:
# 2.2 Dataset形式に変換
train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)

print(f"Train dataset: {train_dataset}")
print(f"Valid dataset: {valid_dataset}")

## Section 3: モデルセットアップ

In [ ]:
# 3.1 量子化設定
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)
print("量子化設定完了")

In [ ]:
# 3.2 モデル・トークナイザーロード
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"モデルロード中: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# パディングトークン設定
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print("モデルロード完了")
print_memory()

In [ ]:
# 3.3 LoRA設定
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 量子化学習用に準備
model = prepare_model_for_kbit_training(model)

# LoRA設定
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

# PEFTモデル作成
model = get_peft_model(model, lora_config)

# 学習可能パラメータ数を表示
model.print_trainable_parameters()
print_memory()

## Section 4: データ前処理

In [ ]:
# 4.1 プロンプトテンプレート
SYSTEM_PROMPT = """あなたは渋谷エリアの地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
座標情報がある場合は必ず含めてください。
数値データがある場合は具体的な数字を使って回答してください。"""

def format_prompt(example):
    """Alpaca形式のデータをチャット形式に変換"""
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    
    if input_text:
        user_content = f"{instruction}\n\n{input_text}"
    else:
        user_content = instruction
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output}
    ]
    
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

# テスト
sample_formatted = format_prompt(train_data[0])
print("フォーマット例:")
print(sample_formatted[:500])

In [ ]:
# 4.2 データセットの前処理
def preprocess_function(examples):
    """バッチ処理用の前処理関数"""
    texts = []
    for i in range(len(examples["instruction"])):
        example = {
            "instruction": examples["instruction"][i],
            "input": examples["input"][i] if "input" in examples else "",
            "output": examples["output"][i]
        }
        texts.append(format_prompt(example))
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors=None
    )
    
    # labelsはinput_idsと同じ（Causal LM）
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

print("データセット前処理中...")
train_dataset_processed = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Train"
)

valid_dataset_processed = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names,
    desc="Valid"
)

print(f"処理後 Train: {train_dataset_processed}")
print(f"処理後 Valid: {valid_dataset_processed}")

## Section 5: 学習実行

In [ ]:
# 5.1 学習設定
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",  # wandbなどを使わない
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

# Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("学習設定完了")

In [ ]:
# 5.2 Trainer作成
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_processed,
    eval_dataset=valid_dataset_processed,
    data_collator=data_collator,
)

print("Trainer作成完了")
print_memory()

In [ ]:
# 5.3 学習実行
print("="*50)
print("学習開始")
print("="*50)

import time
start_time = time.time()

try:
    train_result = trainer.train()
    
    elapsed = time.time() - start_time
    print(f"\n学習完了: {elapsed/60:.1f}分")
    print(f"Train Loss: {train_result.training_loss:.4f}")
    
except Exception as e:
    print(f"エラー: {e}")
    raise

print_memory()

In [ ]:
# 5.4 学習曲線の可視化
import matplotlib.pyplot as plt

# 学習ログから損失を抽出
logs = trainer.state.log_history

train_loss = [(l['step'], l['loss']) for l in logs if 'loss' in l and 'eval_loss' not in l]
eval_loss = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(10, 5))

if train_loss:
    steps, losses = zip(*train_loss)
    ax.plot(steps, losses, label='Train Loss', alpha=0.7)

if eval_loss:
    steps, losses = zip(*eval_loss)
    ax.plot(steps, losses, label='Eval Loss', marker='o')

ax.set_xlabel('Steps')
ax.set_ylabel('Loss')
ax.set_title('Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curve.png", dpi=150)
plt.show()

print(f"学習曲線保存: {OUTPUT_DIR}/training_curve.png")

## Section 6: モデル保存

In [ ]:
# 6.1 LoRAアダプタ保存
print("モデル保存中...")

# LoRAアダプタのみ保存（ベースモデルは含まない）
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"保存完了: {OUTPUT_DIR}")

# 保存ファイル確認
import os
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"  {f}: {size/1024:.1f} KB")

In [ ]:
# 6.2 学習メタデータ保存
import json
from datetime import datetime

training_metadata = {
    "timestamp": datetime.now().isoformat(),
    "base_model": MODEL_NAME,
    "lora_config": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT
    },
    "training_config": {
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRADIENT_ACCUMULATION,
        "learning_rate": LEARNING_RATE,
        "max_seq_length": MAX_SEQ_LENGTH
    },
    "data": {
        "train_samples": len(train_data),
        "valid_samples": len(valid_data)
    },
    "results": {
        "final_train_loss": train_result.training_loss if 'train_result' in dir() else None,
        "training_time_min": elapsed / 60 if 'elapsed' in dir() else None
    }
}

meta_path = f"{OUTPUT_DIR}/training_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)

print(f"メタデータ保存: {meta_path}")

## Section 7: 簡易動作確認

In [ ]:
# 7.1 推論テスト
print("=== 推論テスト ===")

model.eval()

test_questions = [
    "渋谷駅に最も近いコンビニはどこですか？",
    "渋谷駅周辺のカフェを教えてください",
    "渋谷駅の東側と西側、どちらにカフェが多いですか？"
]

def generate_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # アシスタントの回答部分を抽出
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    return response

for q in test_questions:
    print(f"\nQ: {q}")
    answer = generate_answer(q)
    print(f"A: {answer[:300]}..." if len(answer) > 300 else f"A: {answer}")
    clear_memory()

## Section 8: 完了

In [ ]:
# 8.1 サマリー
print("="*50)
print("QLoRAファインチューニング完了")
print("="*50)

print(f"\nベースモデル: {MODEL_NAME}")
print(f"出力ディレクトリ: {OUTPUT_DIR}")
print(f"学習データ: {len(train_data)}件")
print(f"学習時間: {elapsed/60:.1f}分" if 'elapsed' in dir() else "")

print(f"\n次のステップ: finetuning_03_evaluation.ipynb で評価を実行")

In [ ]:
# 8.2 メモリ解放
del model
del trainer
clear_memory()
print("メモリ解放完了")
print_memory()